<!--nav--> [🗺 Learning path](README.md) · **4/49** · ◀ [Simple MultiGPU Benchmark](./Simple_MultiGPU_Benchmark.ipynb) · [Modern Full FineTuning PostTraining](./Modern_Full_FineTuning_PostTraining.ipynb) ▶

# LoRA vs QLoRA vs Full Fine-Tuning — Side by Side

## Three Ways to Fine-Tune an LLM

| Method | What it does | GPU Memory | Trainable Params |
|--------|-------------|------------|------------------|
| **Full Fine-Tuning** | Updates ALL weights | ~12-14 GB | 100% (1.1B) |
| **LoRA** | Small adapters on frozen model | ~6-8 GB | ~1% (11M) |
| **QLoRA** | LoRA + 4-bit quantized model | ~3-5 GB | ~1% (11M) |

### How LoRA Works
Instead of updating a huge weight matrix **W** (4096×4096 = 16M params), LoRA decomposes the update:
```
W_new = W_frozen + A @ B
A is (4096 x r), B is (r x 4096), r = 16 → only 131K params instead of 16M
```

### How QLoRA Extends This
QLoRA = LoRA + 4-bit quantized base model. Frozen weights in 4-bit, training in 16-bit on adapters only.

### This Notebook
We train all 3 methods on the **same model** (TinyLlama 1.1B) and **same data**, then compare:
- GPU memory used
- Training speed
- Output quality

**We push each method to use as much of the T4's 15GB as possible** with large batch sizes and sequence lengths.

---
**Runtime:** T4 GPU (Runtime → Change runtime type → T4 GPU)

## Step 1: Install & Check GPU

In [ ]:
!pip install -q transformers trl datasets peft accelerate bitsandbytes

In [ ]:
import torch
import gc
import time

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    total_memory = torch.cuda.get_device_properties(0).total_memory / 1e9
    print("GPU Memory: %.1f GB" % total_memory)
else:
    raise RuntimeError("No GPU! Go to Runtime > Change runtime type > T4 GPU")

def gpu_mem():
    """Return current GPU memory usage in GB."""
    return torch.cuda.memory_allocated() / 1e9

def gpu_report(label):
    used = torch.cuda.memory_allocated() / 1e9
    reserved = torch.cuda.memory_reserved() / 1e9
    print("%s -> Allocated: %.2f GB | Reserved: %.2f GB" % (label, used, reserved))

def clear_gpu():
    """Free GPU memory between experiments."""
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

## Step 2: Prepare Training Data

We use the **Alpaca** instruction-tuning dataset — 52K instruction/response pairs.
We take 1000 examples to keep training fast while still being meaningful.

In [ ]:
from datasets import load_dataset

dataset = load_dataset("tatsu-lab/alpaca", split="train")
dataset = dataset.shuffle(seed=42).select(range(1000))

MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
MAX_SEQ_LEN = 512  # longer sequences = more GPU usage

print("Dataset: %d examples" % len(dataset))
print("Model:", MODEL_NAME)
print("Max sequence length:", MAX_SEQ_LEN)
print()
print("Example:")
print("  Instruction:", dataset[0]["instruction"][:100])
print("  Output:", dataset[0]["output"][:100])

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def format_example(example):
    """Format Alpaca examples into chat format."""
    instruction = example["instruction"]
    inp = example.get("input", "")
    output = example["output"]

    if inp:
        prompt = instruction + "\n\n" + inp
    else:
        prompt = instruction

    messages = [
        {"role": "user", "content": prompt},
        {"role": "assistant", "content": output},
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False)
    return {"text": text}

formatted_dataset = dataset.map(format_example, remove_columns=dataset.column_names)
print("Formatted %d examples" % len(formatted_dataset))
print("\nSample text (first 200 chars):")
print(formatted_dataset[0]["text"][:200])

## Step 3: Shared Training Config

We keep hyperparameters identical across all 3 methods for a fair comparison.
Batch sizes are tuned to **maximize GPU usage** for each method.

In [ ]:
from transformers import TrainerCallback

# We'll store results from all 3 methods here
results = {}

EPOCHS = 1
LEARNING_RATE = 2e-4
LOGGING_STEPS = 10
MAX_STEPS = 50  # enough to see trends, fast enough to compare all 3

# Batch sizes tuned per method to max out 15GB T4
# Full FT: model(2.2GB) + grads(2.2GB) + optimizer(8.8GB) = ~13GB -> batch=2
# LoRA:    model(2.2GB) + small grads + small optimizer = ~4GB  -> batch=8
# QLoRA:   model(0.7GB) + small grads + small optimizer = ~2GB  -> batch=16
BATCH_SIZES = {
    "full": 2,
    "lora": 8,
    "qlora": 16,
}

# Gradient accumulation to equalize effective batch size
GRAD_ACCUM = {
    "full": 8,    # effective batch = 2 * 8 = 16
    "lora": 2,    # effective batch = 8 * 2 = 16
    "qlora": 1,   # effective batch = 16 * 1 = 16
}

class MemoryTracker(TrainerCallback):
    """Track peak GPU memory during training."""
    def __init__(self):
        self.peak_mb = 0
        self.losses = []
    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs and "loss" in logs:
            self.losses.append((state.global_step, logs["loss"]))
        mem = torch.cuda.max_memory_allocated() / 1e6
        if mem > self.peak_mb:
            self.peak_mb = mem

print("Config ready.")
print("Each method trains for %d steps on %d examples" % (MAX_STEPS, len(formatted_dataset)))
print("Effective batch size: 16 for all methods")
print()
print("Per-method batch sizes (to fill 15GB):")
for k, v in BATCH_SIZES.items():
    print("  %s: batch=%d x grad_accum=%d = effective %d" % (k, v, GRAD_ACCUM[k], v * GRAD_ACCUM[k]))

---
# METHOD 1: Full Fine-Tuning

Updates **every single weight** in the model (1.1B parameters).

**Memory breakdown:**
- Model weights (fp16): ~2.2 GB
- Gradients (fp16): ~2.2 GB
- Optimizer states (AdamW, fp32): ~8.8 GB
- **Total: ~13+ GB** → batch_size=2 to fit in 15GB

In [ ]:
from transformers import AutoModelForCausalLM
from trl import SFTConfig, SFTTrainer

clear_gpu()
print("=" * 60)
print("METHOD 1: FULL FINE-TUNING")
print("=" * 60)

# Load model in fp16 (2.2GB)
model_full = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto",
)
gpu_report("After loading model")

total_params = sum(p.numel() for p in model_full.parameters())
trainable_params = sum(p.numel() for p in model_full.parameters() if p.requires_grad)
print("Total params: %dM" % (total_params / 1e6))
print("Trainable: %dM (%.1f%%)" % (trainable_params / 1e6, 100.0 * trainable_params / total_params))

training_args_full = SFTConfig(
    output_dir="./output_full",
    max_steps=MAX_STEPS,
    per_device_train_batch_size=BATCH_SIZES["full"],
    gradient_accumulation_steps=GRAD_ACCUM["full"],
    learning_rate=LEARNING_RATE,
    logging_steps=LOGGING_STEPS,
    bf16=True,                 # bf16 avoids grad scaler issues
    gradient_checkpointing=True,  # saves ~40% memory
    max_length=MAX_SEQ_LEN,
    report_to="none",
    save_strategy="no",
    dataset_text_field="text",
)

tracker_full = MemoryTracker()

trainer_full = SFTTrainer(
    model=model_full,
    args=training_args_full,
    train_dataset=formatted_dataset,
    processing_class=tokenizer,
    callbacks=[tracker_full],
)

print("\nTraining...")
start_time = time.time()
trainer_full.train()
full_time = time.time() - start_time

gpu_report("After training")
print("Peak GPU memory: %.2f GB" % (tracker_full.peak_mb / 1000))
print("Training time: %.1f seconds" % full_time)

results["full"] = {
    "peak_gpu_gb": tracker_full.peak_mb / 1000,
    "time_sec": full_time,
    "trainable_params": trainable_params,
    "total_params": total_params,
    "losses": tracker_full.losses,
    "batch_size": BATCH_SIZES["full"],
}

# Generate a test response
model_full.eval()
test_prompt = "Explain what an API is in simple terms."
messages = [{"role": "user", "content": test_prompt}]
text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(text, return_tensors="pt").to(model_full.device)
with torch.no_grad():
    out = model_full.generate(**inputs, max_new_tokens=150, temperature=0.7, do_sample=True)
results["full"]["response"] = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()
print("\nTest response:", results["full"]["response"][:200])

# Free memory for next method
del model_full, trainer_full
clear_gpu()
print("\nGPU cleared for next method.")

---
# METHOD 2: LoRA (Low-Rank Adaptation)

Freezes the base model, adds small trainable adapters to attention layers.

**Memory breakdown:**
- Model weights (fp16, frozen): ~2.2 GB
- LoRA adapters + gradients: ~0.1 GB
- Optimizer (only for adapters): ~0.2 GB
- **Total: ~4 GB** → batch_size=8 to push toward 15GB

In [ ]:
from peft import LoraConfig, get_peft_model

clear_gpu()
print("=" * 60)
print("METHOD 2: LoRA")
print("=" * 60)

model_lora = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto",
)
gpu_report("After loading model")

# Apply LoRA — target ALL linear layers for maximum adaptation
lora_config = LoraConfig(
    r=64,                    # rank 64 (higher = more capacity, more memory)
    lora_alpha=128,          # scaling factor
    lora_dropout=0.05,
    target_modules=[         # target ALL attention + MLP layers
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    bias="none",
    task_type="CAUSAL_LM",
)

model_lora = get_peft_model(model_lora, lora_config)
model_lora.print_trainable_parameters()
gpu_report("After applying LoRA")

trainable_params_lora = sum(p.numel() for p in model_lora.parameters() if p.requires_grad)
total_params_lora = sum(p.numel() for p in model_lora.parameters())

training_args_lora = SFTConfig(
    output_dir="./output_lora",
    max_steps=MAX_STEPS,
    per_device_train_batch_size=BATCH_SIZES["lora"],
    gradient_accumulation_steps=GRAD_ACCUM["lora"],
    learning_rate=LEARNING_RATE,
    logging_steps=LOGGING_STEPS,
    bf16=True,                 # bf16 avoids grad scaler issues
    gradient_checkpointing=True,
    max_length=MAX_SEQ_LEN,
    report_to="none",
    save_strategy="no",
    dataset_text_field="text",
)

tracker_lora = MemoryTracker()

trainer_lora = SFTTrainer(
    model=model_lora,
    args=training_args_lora,
    train_dataset=formatted_dataset,
    processing_class=tokenizer,
    callbacks=[tracker_lora],
)

print("\nTraining...")
start_time = time.time()
trainer_lora.train()
lora_time = time.time() - start_time

gpu_report("After training")
print("Peak GPU memory: %.2f GB" % (tracker_lora.peak_mb / 1000))
print("Training time: %.1f seconds" % lora_time)

results["lora"] = {
    "peak_gpu_gb": tracker_lora.peak_mb / 1000,
    "time_sec": lora_time,
    "trainable_params": trainable_params_lora,
    "total_params": total_params_lora,
    "losses": tracker_lora.losses,
    "batch_size": BATCH_SIZES["lora"],
    "lora_r": 64,
}

# Generate test response
model_lora.eval()
inputs = tokenizer(
    tokenizer.apply_chat_template([{"role": "user", "content": test_prompt}], tokenize=False, add_generation_prompt=True),
    return_tensors="pt"
).to(model_lora.device)
with torch.no_grad():
    out = model_lora.generate(**inputs, max_new_tokens=150, temperature=0.7, do_sample=True)
results["lora"]["response"] = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()
print("\nTest response:", results["lora"]["response"][:200])

del model_lora, trainer_lora
clear_gpu()
print("\nGPU cleared for next method.")

---
# METHOD 3: QLoRA (Quantized LoRA)

Same LoRA adapters, but the base model is loaded in **4-bit** precision.

**Memory breakdown:**
- Model weights (4-bit, frozen): ~0.7 GB
- LoRA adapters + gradients: ~0.1 GB
- Optimizer (only for adapters): ~0.2 GB
- **Total: ~2 GB** → batch_size=16 to push toward 15GB

With QLoRA you could fit a **7B model** on a T4. We use 1.1B with huge batches instead.

In [ ]:
from transformers import BitsAndBytesConfig

clear_gpu()
print("=" * 60)
print("METHOD 3: QLoRA (4-bit quantized + LoRA)")
print("=" * 60)

# 4-bit quantization config (the "Q" in QLoRA)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",           # normalized float 4-bit
    bnb_4bit_compute_dtype=torch.bfloat16, # compute in bf16 (must match training)
    bnb_4bit_use_double_quant=True,        # double quantization saves more memory
)

model_qlora = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
)
gpu_report("After loading 4-bit model")

# Same LoRA config as before
qlora_config = LoraConfig(
    r=64,
    lora_alpha=128,
    lora_dropout=0.05,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    bias="none",
    task_type="CAUSAL_LM",
)

model_qlora = get_peft_model(model_qlora, qlora_config)
model_qlora.print_trainable_parameters()
gpu_report("After applying LoRA on 4-bit model")

trainable_params_qlora = sum(p.numel() for p in model_qlora.parameters() if p.requires_grad)
total_params_qlora = sum(p.numel() for p in model_qlora.parameters())

# IMPORTANT: Use bf16=True (not fp16) for QLoRA — bitsandbytes 4-bit uses bfloat16
# fp16 grad scaler is NOT compatible with bfloat16 tensors
training_args_qlora = SFTConfig(
    output_dir="./output_qlora",
    max_steps=MAX_STEPS,
    per_device_train_batch_size=BATCH_SIZES["qlora"],
    gradient_accumulation_steps=GRAD_ACCUM["qlora"],
    learning_rate=LEARNING_RATE,
    logging_steps=LOGGING_STEPS,
    bf16=True,                 # <-- bf16 for QLoRA, NOT fp16
    gradient_checkpointing=True,
    max_length=MAX_SEQ_LEN,
    report_to="none",
    save_strategy="no",
    dataset_text_field="text",
)

tracker_qlora = MemoryTracker()

trainer_qlora = SFTTrainer(
    model=model_qlora,
    args=training_args_qlora,
    train_dataset=formatted_dataset,
    processing_class=tokenizer,
    callbacks=[tracker_qlora],
)

print("\nTraining...")
start_time = time.time()
trainer_qlora.train()
qlora_time = time.time() - start_time

gpu_report("After training")
print("Peak GPU memory: %.2f GB" % (tracker_qlora.peak_mb / 1000))
print("Training time: %.1f seconds" % qlora_time)

results["qlora"] = {
    "peak_gpu_gb": tracker_qlora.peak_mb / 1000,
    "time_sec": qlora_time,
    "trainable_params": trainable_params_qlora,
    "total_params": total_params_qlora,
    "losses": tracker_qlora.losses,
    "batch_size": BATCH_SIZES["qlora"],
    "lora_r": 64,
}

# Generate test response
model_qlora.eval()
inputs = tokenizer(
    tokenizer.apply_chat_template([{"role": "user", "content": test_prompt}], tokenize=False, add_generation_prompt=True),
    return_tensors="pt"
).to(model_qlora.device)
with torch.no_grad():
    out = model_qlora.generate(**inputs, max_new_tokens=150, temperature=0.7, do_sample=True)
results["qlora"]["response"] = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()
print("\nTest response:", results["qlora"]["response"][:200])

del model_qlora, trainer_qlora
clear_gpu()
print("\nGPU cleared.")

---
# Results: Side-by-Side Comparison

In [ ]:
import matplotlib.pyplot as plt

methods = ["full", "lora", "qlora"]
labels = ["Full Fine-Tune", "LoRA", "QLoRA"]
colors = ["#f87171", "#818cf8", "#34d399"]

plt.style.use('dark_background')
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Chart 1: GPU Memory
mem_vals = [results[m]["peak_gpu_gb"] for m in methods]
bars = axes[0].bar(labels, mem_vals, color=colors, edgecolor="white", linewidth=0.5)
axes[0].axhline(y=15, color="#f59e0b", linestyle="--", linewidth=1, label="T4 limit (15GB)")
axes[0].set_title("Peak GPU Memory (GB)", fontsize=14)
axes[0].set_ylabel("GB")
axes[0].legend()
for bar, val in zip(bars, mem_vals):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2,
                 "%.1f" % val, ha="center", fontsize=12, fontweight="bold")

# Chart 2: Training Time
time_vals = [results[m]["time_sec"] for m in methods]
bars = axes[1].bar(labels, time_vals, color=colors, edgecolor="white", linewidth=0.5)
axes[1].set_title("Training Time (seconds)", fontsize=14)
axes[1].set_ylabel("Seconds")
for bar, val in zip(bars, time_vals):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                 "%.0f" % val, ha="center", fontsize=12, fontweight="bold")

# Chart 3: Loss Curves
for m, label, color in zip(methods, labels, colors):
    losses = results[m]["losses"]
    if losses:
        steps_list = [x[0] for x in losses]
        loss_list = [x[1] for x in losses]
        axes[2].plot(steps_list, loss_list, color=color, linewidth=2, marker="o", markersize=4, label=label)
axes[2].set_title("Training Loss", fontsize=14)
axes[2].set_xlabel("Step")
axes[2].set_ylabel("Loss")
axes[2].legend()
axes[2].grid(True, alpha=0.2)

plt.tight_layout()
plt.show()

In [ ]:
# Trainable parameters comparison
plt.style.use('dark_background')
fig, ax = plt.subplots(figsize=(10, 5))

param_vals = [results[m]["trainable_params"] / 1e6 for m in methods]
bars = ax.bar(labels, param_vals, color=colors, edgecolor="white", linewidth=0.5)
ax.set_title("Trainable Parameters (Millions)", fontsize=14)
ax.set_ylabel("Millions")
for bar, val in zip(bars, param_vals):
    if val > 100:
        text = "%dM" % val
    else:
        text = "%.1fM" % val
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
            text, ha="center", fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()

## Summary Table

In [ ]:
from IPython.display import HTML, display

rows = ""
for m, label, color in zip(methods, labels, colors):
    r = results[m]
    pct = 100.0 * r["trainable_params"] / r["total_params"]
    params_str = "%dM" % (r["trainable_params"] / 1e6) if r["trainable_params"] > 1e8 else "%.1fM" % (r["trainable_params"] / 1e6)
    rows += (
        '<tr>'
        '<td style="color:' + color + ';font-weight:bold;">' + label + '</td>'
        '<td>' + params_str + ' (' + ("%.1f" % pct) + '%)</td>'
        '<td>' + ("%.1f GB" % r["peak_gpu_gb"]) + '</td>'
        '<td>' + ("%.0f sec" % r["time_sec"]) + '</td>'
        '<td>batch=' + str(r["batch_size"]) + '</td>'
        '</tr>'
    )

html = (
    '<table style="width:100%;border-collapse:collapse;font-family:sans-serif;background:#131927;">'
    '<tr style="border-bottom:2px solid #4f46e5;">'
    '<th style="text-align:left;padding:10px;color:#a78bfa;">Method</th>'
    '<th style="text-align:left;padding:10px;color:#a78bfa;">Trainable Params</th>'
    '<th style="text-align:left;padding:10px;color:#a78bfa;">Peak GPU</th>'
    '<th style="text-align:left;padding:10px;color:#a78bfa;">Time</th>'
    '<th style="text-align:left;padding:10px;color:#a78bfa;">Batch Size</th>'
    '</tr>'
    + rows +
    '</table>'
)
display(HTML(html))

## Generated Responses Comparison

In [ ]:
print('Prompt: "%s"' % test_prompt)
print()
for m, label in zip(methods, labels):
    print("%s:" % label)
    print("  %s" % results[m]["response"][:300])
    print()

---
## Key Takeaways

| Question | Answer |
|----------|--------|
| **Which is best?** | Depends on your constraints |
| **Most memory efficient?** | QLoRA — 4-bit model + tiny adapters |
| **Best quality?** | Full fine-tuning (if you have enough GPU + data) |
| **Best tradeoff?** | LoRA — near full-FT quality at fraction of memory |
| **When to use QLoRA?** | When your model doesn't fit in GPU otherwise (7B+ models on T4) |

### The Memory Math
```
Full FT memory  = model + gradients + optimizer = 2x + 2x + 8x = 12x model size
LoRA memory     = model + tiny_grads + tiny_opt = 2x + 0.02x + 0.08x ≈ 2.1x
QLoRA memory    = model/4 + tiny_grads + tiny_opt = 0.5x + 0.02x + 0.08x ≈ 0.6x
```

For a 7B model (14GB in fp16):
- Full FT: ~84 GB → needs A100 80GB
- LoRA: ~30 GB → needs A100 40GB
- QLoRA: ~8 GB → **fits on a free T4!**